## Reference Visual Libraries Cheatsheets
- Matplotlib: https://matplotlib.org/cheatsheets  
- Seaborn: https://seaborn.pydata.org/examples/index.html#cheat-sheets  
- Plotnine (ggplot): https://plotnine.org/reference/  
- Plotly: https://plotly.com/python/plotly-express/

# Graduate Bioinformatics Study Session
## Integrating Data Analysis and Visualization with Pandas, Seaborn, Plotnine, and Plotly

This notebook introduces essential biological datasets and computational workflows for students from diverse backgrounds (biology, computer science, mathematics, or data science). Each exercise is designed to teach both **data-handling techniques** and **biological interpretation**.

After completing the notebook, you will:
- Understand how gene expression, protein modification, and microbiome data are structured and what they represent.
- Use pandas to summarize and merge biological datasets.
- Create informative visualizations with seaborn, plotnine (ggplot), and plotly.
- Interpret biological results without requiring deep prior biological knowledge.

## Problem 1: Gene Expression Variability Across Tissues

### Biological Context
In molecular biology, gene expression refers to how much of a given RNA or protein is produced from a gene. Expression levels differ between tissues: for example, liver genes involved in metabolism may be highly active while brain-specific genes remain inactive there.

Housekeeping genes such as GAPDH or ACTB are essential for basic cell function and tend to have **low variance** across tissues. Highly variable genes indicate **tissue specialization**.

### Dataset Description
- `gene_id`: A unique identifier corresponding to an Ensembl gene record.
- `gene_name`: Human-readable name for the gene.
- `tissue`: The tissue in which expression is measured (e.g., Brain, Heart).
- `expression_tpm`: Gene expression level expressed as TPM (Transcripts Per Million), a normalized RNA-seq measure.

### Student Tasks
1. Compute the mean and variance of expression levels per gene.
2. Identify genes with particularly high or low variance.
3. Visualize results using seaborn and plotnine (ggplot) to compare plotting frameworks.
4. Interpret what tissue variability implies about gene roles.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from plotnine import *
import plotly.express as px

# Step 1: Generate synthetic gene expression dataset
# np.random.seed() ensures reproducibility of random numbers in future reruns.
np.random.seed(42)
tissues = ['Brain','Heart','Liver','Lung','Skin']  # Representative tissues
genes = ['GAPDH','LDHA','PDHA1','PKM','TP53','BRCA1','MYC']  # Mix of metabolic and cancer-associated genes

records = []
for gene in genes:
    for tissue in tissues:
        # For simplicity, we simulate expression values with some tissue variation.
        expression_tpm = abs(np.random.normal(50 + len(gene), 15))  # Normal distribution with slight gene-specific offset.
        records.append([f'ENSG{np.random.randint(100000,999999)}', gene, tissue, expression_tpm])

# Convert to DataFrame for analysis.
expression = pd.DataFrame(records, columns=['gene_id','gene_name','tissue','expression_tpm'])
expression.head()

## Problem 2: Protein Abundance and Post-Translational Modifications (PTMs)

### Biological Context
Proteins are the workhorses of the cell. Their abundance can vary by cell type, and post-translational modifications (PTMs) like phosphorylation or acetylation regulate protein function. A high number of PTMs often indicates dynamic regulation.

### Dataset Description
- `uniprot_id`: Unique identifier from UniProt database.
- `gene_name`: Name of the corresponding gene.
- `abundance`: Quantitative protein level (arbitrary units).
- `ptm_count`: Number of confirmed modification sites.
- `protein_class`: Functional role grouping (signaling, enzyme, regulatory, structural).

### Tasks
1. Compute summary statistics per protein class.
2. Plot correlation between abundance and PTM count using seaborn.
3. Explore distribution of PTM counts using ggplot-based density plots.
4. Explain biological relationships between protein type and modification level.

In [ ]:
import random
classes = ['Enzyme','Regulatory','Structural','Signaling']
proteins = pd.DataFrame({
    'uniprot_id': [f'P{np.random.randint(10000,99999)}' for _ in range(80)],
    'gene_name': [random.choice(['TP53','AKT1','MAPK1','MYH9','LDHA','GAPDH']) for _ in range(80)],
    'abundance': np.abs(np.random.normal(100,35,80)),  # simulated abundance distribution.
    'ptm_count': np.random.poisson(5,80),  # Poisson-distributed PTM frequency.
    'protein_class': [random.choice(classes) for _ in range(80)]
})


## Problem 3: Microbiome Abundance and Host Metadata Integration

### Biological Context
The human microbiome comprises bacterial populations inhabiting body sites like skin, mouth, and gut. Composition and diversity are closely tied to human health, metabolism, and diet.

- **Body site** affects dominant genus composition (gut richer than skin).
- **BMI (Body Mass Index)** may correlate with relative abundances of specific bacterial genera such as *Bacteroides*.

### Dataset Description
- `hmp_microbiome_abundance.csv`: Relative abundance of bacterial genera (percentage of total population per sample).
- `hmp_sample_metadata.csv`: Host metadata including body site and BMI.

### Tasks
1. Merge abundance with metadata using `pandas.merge()`.
2. Compute genus richness using `groupby()` and `nunique()`.
3. Generate visualization comparing microbial diversity per body site.
4. Examine relationships between host BMI and the genus *Bacteroides* using regression.



In [ ]:
from sklearn.decomposition import PCA
samples = [f'S{i:03d}' for i in range(1,25)]
body_sites = ['Gut','Oral','Skin']
genera = ['Bacteroides','Prevotella','Streptococcus','Staphylococcus','Lactobacillus']

# Generate microbiome abundance data; Dirichlet ensures proportions sum to 1 per sample.
abund_records=[]
for s in samples:
    rel = np.random.dirichlet(np.ones(len(genera)))
    for g,a in zip(genera,rel): abund_records.append([s,g,a])
abundance = pd.DataFrame(abund_records,columns=['sample_id','genus','rel_abundance'])

# Create metadata with body site classification and BMI values.
metadata = pd.DataFrame({
    'sample_id': samples,
    'body_site': [np.random.choice(body_sites) for _ in samples],
    'bmi': np.random.uniform(18,35,len(samples))})



## Problem 4: SARS-CoV-2 Spike Protein Structural Data (Public Dataset)

### Biological Context
This dataset retrieves records from the Protein Data Bank (PDB) for SARS-CoV-2 spike protein 3D structures. The spike mediates viral entry into human cells, and structural details show improvements in technology (especially cryo-EM).

### Dataset Description
- `pdb_id`: Protein structure identifier.
- `experimental_method`: Technique used (e.g., Cryo-EM or X-ray crystallography).
- `resolution`: Level of atomic detail (smaller numbers = higher resolution).
- `release_year`: Year structure was published online.

### Tasks
1. Query the RCSB PDB API for spike structures.
2. Parse results and create DataFrame.
3. Plot resolution distribution using seaborn.
4. Plot average resolution trend versus year using ggplot.

In [ ]:
import requests, datetime as dt
# RCSB search query formatted in JSON to find entries containing 'SARS-CoV-2 spike' in title.
url='https://search.rcsb.org/rcsbsearch/v2/query?json={"query":{"type":"terminal","service":"text","parameters":{"attribute":"struct.title","operator":"contains_words","value":"SARS-CoV-2 spike"}},"return_type":"entry","request_options":{"paginate":{"start":0,"rows":15}}}'
response = requests.get(url).json()
ids = [x['identifier'] for x in response['result_set']]

# For simplicity, get metadata for 10 entries.
records=[]
for pdb in ids[:10]:
    info=requests.get(f'https://data.rcsb.org/rest/v1/core/entry/{pdb}').json()
    method=info.get('experimental_method','NA')
    resolution=info.get('rcsb_entry_info',{}).get('resolution_combined',[None])[0]
    release=info.get('rcsb_accession_info',{}).get('initial_release_date','2020-01-01')
    year=dt.datetime.strptime(release[:10],'%Y-%m-%d').year
    records.append([pdb,method,resolution,year])
pdb_df=pd.DataFrame(records,columns=['pdb_id','experimental_method','resolution','release_year'])



## Review Quiz
1. Which pandas methods can calculate group averages?  
2. Why are metabolic genes expected to show co-expression patterns?  
3. What does kernel density estimation reveal in ggplot density plots?  
5. What is the biological meaning of low vs high structure resolution values?  
6. Name advantages of using seaborn vs plotnine.

## Tips to check if you understood the concepts
- Annotate each plot with a one-sentence biological conclusion.
- Explain computation steps (mean, variance).
- Resize and color-tune plots interactively to illustrate parameter effects and visual clarity.